# REF Academic Paper Evaluator
**Copyright: Jac Ison 2025**

### Intended use
Desktop/notebook tool to assist academic reviewers in pre-assessing research papers against REF2029 criteria (originality, significance, rigour). This is **decision support, not a replacement for expert review** — see `5.3 Ethical and Institutional Implications` in the accompanying dissertation.

### How it works
1. Upload one or more PDF or TXT papers and click **Evaluate**.
2. Text is extracted, the reference list is removed, and the body is split into ~1300-word segments (≈1500 NLTK tokens, as in the dissertation).
3. Up to `max_segments` segments are scored by an LLM (Cohere `command-a-03-2025`) on originality, significance and rigour (1–4), using a strict REF-reviewer system prompt and structured JSON output.
4. A calibration step compresses over-generous or internally inconsistent scores (dissertation §3.5).
5. An impact-keyword heuristic applies a small weighting factor (1.00–1.15).
6. Results are mapped to a REF band (1*–4*, with High/Middle/Low sub-bands), cached, and appended to `ref_results.csv`.

The logic lives in the `ref_evaluator` package next to this notebook, so it can be tested (`pytest`) and used from scripts; this notebook is only the interface.

### Limitations (dissertation §3.6 / §7)
- Does not replace human judgement.
- Textual proxies for originality/rigour may miss contribution types (datasets, software, non-textual outputs).
- Results may not generalise beyond life sciences without domain adaptation.
- Only the first `max_segments` segments of long papers are scored by default. **You will be warned if a paper is truncated.**
- Temperature 0 and a fixed seed make runs as repeatable as the API allows, but upstream model updates can still shift outputs.

## Setup (run once)

1. Copy `.env.example` to `.env` in the same folder as this notebook (`.env` is already git-ignored — never commit it).
2. Replace the placeholder with your Cohere key:
   ```
   COHERE_API_KEY=your-actual-key-here
   ```
3. Run the cells below in order.

If you've previously used this tool with a hardcoded key in any notebook, `.env` file or `.zip` you've shared, **rotate that key in your Cohere dashboard**.

In [ ]:
%pip install -r requirements.txt --quiet

## Config

Everything you're likely to want to tune is a field on `Config` — see `ref_evaluator/config.py` for the full list. Changing any setting that affects scores (model, prompt, temperature, seed, segment size, segment cap, impact rules) automatically gives new cache entries, so old results are never served for new settings. If you change the calibration rules in code, bump `scoring_version`.

In [ ]:
import os

import cohere
from dotenv import load_dotenv

from ref_evaluator import Config, ResultCache

load_dotenv()
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
if not COHERE_API_KEY:
    raise RuntimeError(
        "COHERE_API_KEY not found. Create a .env file alongside this notebook "
        "containing: COHERE_API_KEY=your-actual-key-here"
    )

config = Config(
    max_segments=5,     # set to None to score the whole paper
)
cache = ResultCache("ref_cache")
RESULTS_CSV = "ref_results.csv"
client = cohere.ClientV2(api_key=COHERE_API_KEY)

## Interface

Choose one or more files, then click **Evaluate**. Each file is processed and reported independently, and every batch is appended to `ref_results.csv` for comparison with expert ratings.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

from ref_evaluator import ScoringError, evaluate_with_cache, write_results_csv

upload_widget = widgets.FileUpload(accept=".pdf,.txt", multiple=True)
evaluate_button = widgets.Button(description="▶️ Evaluate", button_style="primary")
clear_cache_button = widgets.Button(description="🧹 Clear Cache", button_style="warning")
output = widgets.Output()


def _on_clear_cache(_):
    output.clear_output()
    with output:
        cache.clear()
        print("🧹 Cache cleared successfully.")


def _on_evaluate(_):
    output.clear_output()
    with output:
        if not upload_widget.value:
            print("Please choose at least one file.")
            return

        results = []
        for uploaded_file in upload_widget.value:
            file_name = uploaded_file["name"]
            print(f"📄 Evaluating paper: {file_name}")
            try:
                result, from_cache = evaluate_with_cache(
                    client, file_name, bytes(uploaded_file["content"]), config, cache
                )
            except ScoringError as e:
                print(f"❌ {e}\nStopping: fix this before evaluating more files.")
                break
            except Exception as e:
                print(f"❌ Evaluation failed for {file_name}: {e}")
            else:
                if from_cache:
                    print("✅ Loaded cached result (same file + same settings).")
                    if result.truncated:
                        print(f"⚠️ Only {result.segments_evaluated} of {result.segment_count} segments were scored.")
                    print("Final REF Rating:", result.final_ref)
                results.append(result)
            print("-" * 60)

        if results:
            print(f"📊 Results appended to {write_results_csv(results, RESULTS_CSV)}")
        upload_widget.value = ()


display(widgets.HTML("<h3>📂 Upload academic paper(s) (.pdf or .txt):</h3>"))
display(widgets.HBox([upload_widget, evaluate_button, clear_cache_button]), output)
evaluate_button.on_click(_on_evaluate)
clear_cache_button.on_click(_on_clear_cache)